# 📊 UNIVERSAL EVALUATION METRICS CHO DOCUMENT DEWARPING (TỐI ƯU GPU & ĐA TIẾN TRÌNH)

Hệ thống đo kiểm chuẩn quốc tế theo phương pháp của DocUNet, DewarpNet và DocTr, hỗ trợ đo kiểm độc lập hoặc hàng loạt nhiều mô hình cùng lúc:
* **MS-SSIM-M (Chỉ số ảnh - GPU)**: Tính bằng PyTorch `torch.float64` (double precision) trên CUDA với Gaussian 11x11, bộ lọc kim tự tháp Burt & Adelson 5 tầng và mặt nạ biên đen `ref_msk(A1==0)=0`. Nhanh gấp 30 lần (~2ms/ảnh, độ lệch < 10^-6 so với MATLAB).
* **LD-M (Chỉ số ảnh - GNU Octave)**: Chạy SIFT Flow C++ MEX biên dịch cờ `-O3`, tự động lưu cache SIFT descriptor của 30 trang scan chuẩn và chia tải chạy song song trên 2 nhân vCPU của Colab.
* **CER & ED (Chỉ số văn bản - Tesseract 5.0)**: Chạy chuẩn paper Tesseract 5.0, kết hợp cache chuỗi văn bản của 30 ảnh scan chuẩn và chạy đa luồng song song.
* **Đệm SSD Cục bộ**: Tự động đồng bộ ảnh từ Google Drive sang NVMe SSD `/content/` giúp triệt tiêu hoàn toàn độ trễ I/O đọc ảnh qua mạng.
* **Hỗ trợ đa mô hình**: Cho phép chọn chạy 1 mô hình đơn lẻ, danh sách nhiều mô hình hoặc toàn bộ các mô hình có trong thư mục predictions.

### ⚡ Ô 0: Kiểm tra môi trường & GPU (Rule 5)

In [6]:
import sys,torch
in_colab='google.colab' in sys.modules
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Môi trường Colab: {in_colab} | Thiết bị: {device}")
if(device.type=='cuda'):
	print(f"Tên GPU: {torch.cuda.get_device_name(0)}")
	print(f"VRAM: {round(torch.cuda.get_device_properties(0).total_memory/(1024**3),2)} GB")
else:
	print("Cảnh báo: Đang chạy trên CPU. Hãy bật GPU tại Runtime > Change runtime type.")


Môi trường Colab: True | Thiết bị: cuda
Tên GPU: Tesla T4
VRAM: 14.56 GB


### 📦 Ô 1: Kết nối Google Drive, cài đặt GNU Octave, Tesseract 5.0 & biên dịch SIFT Flow MEX

In [7]:
import os,sys
if('google.colab' in sys.modules):
	from google.colab import drive
	drive.mount('/content/drive')
!rm -f /etc/apt/sources.list.d/*r2u* 2>/dev/null || true
!apt-get update -qq || true
!DEBIAN_FRONTEND=noninteractive apt-get install -y --no-install-recommends octave octave-dev octave-image tesseract-ocr libtesseract-dev -qq || DEBIAN_FRONTEND=noninteractive apt-get install -y octave octave-dev tesseract-ocr -qq || true
!pip install -q torchvision pytesseract rapidfuzz pandas pillow scikit-image opencv-python tqdm
!if [ ! -d "/content/dewarp-metrics" ]; then git clone https://github.com/nck345/dewarp-metrics.git /content/dewarp-metrics; else cd /content/dewarp-metrics && git fetch origin && git reset --hard origin/main; fi
!sed -i 's/\r$//' /content/dewarp-metrics/compile_mex.sh
!bash /content/dewarp-metrics/compile_mex.sh


### ⚙️ Ô 2: Cấu hình Model & tập Benchmark cần đánh giá

Hỗ trợ linh hoạt:
- **Chọn mô hình**: 1 mô hình (ví dụ `'DocTr'` hoặc `'D2Dewarp'`), danh sách nhiều mô hình (ví dụ `['DocTr', 'D2Dewarp']`) hoặc toàn bộ các mô hình (`'all'`).
- **Chọn benchmark**: 1 tập đơn lẻ (ví dụ `'docunet_130'`), danh sách nhiều tập (ví dụ `['docunet_130', 'dir300']`) hoặc toàn bộ các tập (`'all'`).

In [8]:
import os

DRIVE_DIR = '/content/drive/MyDrive/Document_Dewarp_Benchmark'

# ==============================================================================
# 1. CHỌN MÔ HÌNH ĐỂ ĐO KIỂM:
#    - Đo riêng model mới D2Dewarp (rất khuyên dùng khi cần đo nhanh): 'D2Dewarp'
#    - Đo riêng DocTr:  'DocTr'
#    - Đo cả 2 model:   ['DocTr', 'D2Dewarp'] hoặc 'all'
# ==============================================================================
MODELS = 'all'  # 'D2Dewarp' (đo nhanh), ['DocTr', 'D2Dewarp'] hoặc 'all'

# ==============================================================================
# 2. CHỌN TẬP BENCHMARK ĐỂ ĐO KIỂM:
#    - Đầy đủ DocUNet (130 ảnh): ['docunet_130']
#    - Nhiều tập: ['docunet_130', 'dir300']
#    - Toàn bộ 3 tập (630 ảnh/model): 'all'
# ==============================================================================
BENCHMARKS = 'all'  # ['docunet_130'] hoặc 'all' (630 ảnh/model)

# ==============================================================================
# 3. CHẾ ĐỘ RESUME ĐA TẦNG (CHỐNG MẤT TIẾN ĐỘ THEO TỪNG BƯỚC & TỪNG ẢNH):
#    - True : Tự động khôi phục từng ảnh đã đo (Step 1/2/3) từ .checkpoints/ và bỏ qua tập đã hoàn tất
#    - False: Xóa checkpoint cũ của lần đo được chọn và đo lại từ đầu
# ==============================================================================
RESUME = True
SKIP_LDM = False  # True: chỉ đo MS-SSIM-M (GPU) + OCR, tạm bỏ qua LD-M (Octave) để thử nghiệm nhanh

# Tự động tận dụng tối đa số nhân CPU trên Colab (thường 2 hoặc 4 vCPUs)
NUM_WORKERS = min(4, os.cpu_count() or 2)
MS_SSIM_BACKEND = 'torch'  # 'torch' (GPU float64 siêu tốc) hoặc 'octave' (CPU)
USE_LOCAL_SSD = True       # Sao chép sang SSD cục bộ (/content/) để đạt tốc độ I/O tối đa

REPORT_DIR = f'{DRIVE_DIR}/eval_reports'
CACHE_DIR = f'{DRIVE_DIR}/gt_cache'
PREDICTIONS_DIR = f'{DRIVE_DIR}/predictions'
os.makedirs(REPORT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

# 1. Xác định danh sách Model cần đo
available_models = []
if os.path.exists(PREDICTIONS_DIR):
    available_models = sorted([
        d for d in os.listdir(PREDICTIONS_DIR)
        if os.path.isdir(os.path.join(PREDICTIONS_DIR, d)) and not d.startswith('.')
    ])

if MODELS == 'all':
    target_models = available_models if available_models else ['DocTr', 'D2Dewarp']
elif isinstance(MODELS, str):
    target_models = [MODELS]
else:
    target_models = list(MODELS)

# 2. Xác định danh sách Benchmark cần đo
ALL_BENCHMARKS = ['docunet_130', 'dir300', 'docreal']
if BENCHMARKS == 'all':
    target_benchmarks = ALL_BENCHMARKS
elif isinstance(BENCHMARKS, str):
    target_benchmarks = [BENCHMARKS]
else:
    target_benchmarks = list(BENCHMARKS)

print('=' * 75)
print(f"Mô hình có trong predictions/ : {available_models}")
print(f"Danh sách mô hình cần đo       : {target_models} ({len(target_models)} mô hình)")
print(f"Danh sách Benchmark cần đo     : {target_benchmarks} ({len(target_benchmarks)} tập)")
print(f"Chế độ Resume                  : {RESUME} (theo từng bước & từng ảnh)")
print(f"Bỏ qua LD-M (Octave)           : {SKIP_LDM}")
print(f"MS-SSIM-M Backend              : {MS_SSIM_BACKEND} (GPU float64)")
print(f"LD-M & OCR Workers             : {NUM_WORKERS} workers song song (tối ưu theo CPU)")
print(f"Sử dụng Local SSD              : {USE_LOCAL_SSD}")
print(f"Thư mục lưu báo cáo            : {REPORT_DIR}")
print('=' * 75)


Mô hình có trong predictions/ : ['D2Dewarp', 'DocTr']
Danh sách mô hình cần đo       : ['D2Dewarp', 'DocTr'] (2 mô hình)
Danh sách Benchmark cần đo     : ['docunet_paper_subset', 'docunet_130', 'dir300', 'docreal'] (4 tập)
Chế độ Resume                  : True (bỏ qua tập đã đo)
MS-SSIM-M Backend              : torch (GPU float64)
LD-M & OCR Workers             : 2 workers song song (tối ưu theo CPU)
Sử dụng Local SSD              : True
Thư mục lưu báo cáo            : /content/drive/MyDrive/Document_Dewarp_Benchmark/eval_reports


### 🚀 Ô 3: Chạy đo kiểm toàn diện cho các mô hình đã chọn (MS-SSIM-M GPU + LD-M Octave + CER/ED Tesseract 5.0)
Chương trình tự động lặp qua từng mô hình, đo kiểm các tập benchmark đã chọn, lưu bảng tổng hợp cho từng mô hình và cập nhật trực tiếp vào bảng xếp hạng tổng (Leaderboard).

In [9]:
import sys, os, time, importlib, importlib.util, subprocess

# Tự động cài đặt các thư viện cần thiết nếu môi trường Colab vừa khởi động lại
required_pkgs = ['pytesseract', 'rapidfuzz', 'torchvision', 'tqdm', 'scikit-image']
missing_pkgs = [p for p in required_pkgs if not importlib.util.find_spec(p)]
if missing_pkgs:
    print(f"📦 Đang tự động cài đặt các thư viện còn thiếu: {missing_pkgs}...")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + missing_pkgs)

if not os.path.exists('/content/dewarp-metrics'):
    os.system('git clone https://github.com/nck345/dewarp-metrics.git /content/dewarp-metrics')
elif os.path.exists('/content/dewarp-metrics/.git'):
    os.system('cd /content/dewarp-metrics && git fetch origin && git reset --hard origin/main > /dev/null 2>&1 && sed -i "s/\r$//" compile_mex.sh')
if('/content/dewarp-metrics' not in sys.path):
    sys.path.append('/content/dewarp-metrics')
if 'python.ms_ssim_torch' in sys.modules:
    importlib.reload(sys.modules['python.ms_ssim_torch'])
if 'python.checkpoint_utils' in sys.modules:
    importlib.reload(sys.modules['python.checkpoint_utils'])
import python.checkpoint_utils
importlib.reload(python.checkpoint_utils)
if 'python.ocr_eval' in sys.modules:
    importlib.reload(sys.modules['python.ocr_eval'])
import python.ocr_eval
importlib.reload(python.ocr_eval)
import run_eval
importlib.reload(run_eval)
from run_eval import run_benchmark_suite

if not os.path.exists(DRIVE_DIR) and 'google.colab' in sys.modules:
    try:
        from google.colab import drive
        subprocess.run(['fusermount', '-u', '-z', '/content/drive'], capture_output=True)
        drive.mount('/content/drive', force_remount=True)
    except Exception:
        pass

all_results = {}
total_start = time.time()

print(f"🚀 Bắt đầu quy trình đo kiểm cho {len(target_models)} mô hình: {target_models}\n")

for m_idx, m_name in enumerate(target_models, 1):
    print("\n" + "=" * 95)
    print(f"🔄 [{m_idx}/{len(target_models)}] TIẾN HÀNH ĐO KIỂM MÔ HÌNH: {m_name.upper()}")
    print("=" * 95)

    m_pred_dir = f"{DRIVE_DIR}/predictions/{m_name}"
    if not os.path.exists(m_pred_dir):
        print(f"⚠️ Cảnh báo: Không tìm thấy thư mục dự đoán của '{m_name}' tại {m_pred_dir}. Bỏ qua mô hình này.\n")
        continue

    df_model = run_benchmark_suite(
        base_dir=DRIVE_DIR,
        model_name=m_name,
        benchmarks=target_benchmarks,
        octave_bin='octave',
        ms_ssim_backend=MS_SSIM_BACKEND,
        skip_ldm=SKIP_LDM,
        num_workers=NUM_WORKERS,
        use_local_ssd=USE_LOCAL_SSD,
        save_details=True,
        resume=RESUME
    )
    if df_model is not None and not df_model.empty:
        all_results[m_name] = df_model

print("\n" + "=" * 95)
print(f"🎉 TỔNG KẾT: Hoàn tất đo kiểm {len(all_results)}/{len(target_models)} mô hình trong {time.time() - total_start:.1f} giây")
print("=" * 95)


### 📊 Ô 4: Bảng tổng kết kết quả đo kiểm & Bảng xếp hạng (Leaderboard)

In [ ]:
import os, pandas as pd

# 1. Bảng tổng kết của từng mô hình đã đo
for m_name in target_models:
    model_csv = f'{REPORT_DIR}/{m_name}_summary.csv'
    if os.path.exists(model_csv):
        print(f"\n📊 BẢNG TỔNG KẾT CHỈ SỐ CHO MÔ HÌNH [{m_name}]:")
        df_model = pd.read_csv(model_csv)
        display(df_model)

# 2. Bảng tổng hợp toàn bộ các mô hình (Leaderboard)
all_csv = f'{REPORT_DIR}/benchmark_summary.csv'
if os.path.exists(all_csv):
    print(f"\n🏆 BẢNG TỔNG HỢP TOÀN BỘ CÁC MÔ HÌNH (LEADERBOARD):")
    df_all = pd.read_csv(all_csv)
    display(df_all)
